# OCTO Mobile Google Play Review - Exploratory Data Analysis (EDA)

**Author:** Alisha  

**Notebook:** 1 - Data Scrapping & Initial EDA

Notebook ini berisi proses pengambilan data review aplikasi OCTO Mobile dari Google Play, pembersihan data awal, visualisasi eksploratif, dan ekspor dataset hasil scraping.

## Daftar Isi

1. Setup Environment
2. Data Scrapping Google Play Review
3. Data Cleaning dan Feature Engineering
4. Export Dataset
5. Visualisasi EDA

## 1. Setup Environment

Install library yang dibutuhkan untuk scraping, manipulasi data, dan visualisasi.

In [ ]:
!pip -q install google-play-scraper pandas seaborn matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00


## 2. Data Scrapping Google Play Review

Pada tahap ini data review OCTO Mobile diambil bertahap menggunakan continuation token agar lebih stabil untuk data dalam jumlah besar.

In [ ]:
from google_play_scraper import Sort, reviews
import pandas as pd

def scrape_reviews(app_id: str, total_count: int = 5000, batch_size: int = 200):
    """Scrape review Google Play secara bertahap dengan continuation token."""
    all_reviews = []
    token = None

    while len(all_reviews) < total_count:
        remaining = total_count - len(all_reviews)
        current_batch = min(batch_size, remaining)

        result, token = reviews(
            app_id,
            lang="id",
            country="id",
            sort=Sort.NEWEST,
            count=current_batch,
            continuation_token=token,
            filter_score_with=None,
        )

        if not result:
            break

        all_reviews.extend(result)

        # Jika token habis, berarti sudah tidak ada data lanjutan.
        if token is None:
            break

    return pd.DataFrame(all_reviews)

# Ambil data ulasan OCTO Mobile
df_octo = scrape_reviews(
    app_id="id.co.cimbniaga.mobile.android",
    total_count=5000,
    batch_size=200,
    )

print(f"Total review berhasil diambil: {len(df_octo)}")
df_octo.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,41502635-92c9-4157-a5d8-776b6abab7df,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Semua ulasan isinya sama ; Tiba2 nggk bisa dig...,1,0,None,2026-03-10 00:39:19,Mohon maaf atas kendala yang dialami. Apakah d...,2026-03-10 00:45:11,None
1,53272b9e-6fe2-456f-a150-457d526d0ebd,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,CIMB sangat bagus gampang dan mudah buat trans...,5,0,3.1.82,2026-03-09 18:28:46,Terima kasih atas supportnya. Aplikasi OCTO se...,2026-03-09 18:44:51,3.1.82
2,d5f78df1-d97d-4a62-a251-7a7f90b3a197,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,masih ragu nihh buat memulainya karna ada opsi...,1,0,3.1.82,2026-03-09 16:31:33,Mohon maaf atas kendala yang dialami untuk men...,2026-03-09 16:45:31,3.1.82
3,fc3ad107-2a1c-4b63-95ea-35c52ae637f0,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,good,5,0,3.1.74,2026-03-09 16:07:31,Thank you very much for your support! We keep ...,2026-03-09 16:14:54,3.1.74
4,2411ee9d-5e9e-4ef4-a575-f33470e0d2d2,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,jozzzz,5,0,3.1.81,2026-03-09 12:32:02,Terima kasih atas bintang lima dan review nya....,2026-03-09 12:45:51,3.1.81
...,...,...,...,...,...,...,...,...,...,...,...
49995,3bf4bcd4-673c-426c-b633-8abeb5aeea4b,arnen,https://play-lh.googleusercontent.com/a-/ALV-U...,고맙습니다,5,0,2.8.4,2022-05-14 00:37:01,Terima kasih atas supportnya. OCTO Mobile sela...,2022-05-14 08:45:44,2.8.4
49996,12205ab9-26cf-4906-b7fd-7b705c2fcbd1,Refan Maulana,https://play-lh.googleusercontent.com/a-/ALV-U...,Matap,5,0,None,2022-05-14 00:16:31,Terima kasih atas supportnya. OCTO Mobile sela...,2022-05-14 08:45:30,None
49997,bcc30b82-07c5-418d-abee-4a82d8e2d0fc,Gitar_ars,https://play-lh.googleusercontent.com/a-/ALV-U...,Kenapa saat login sering tidak bisa dan tidak ...,2,0,2.8.4,2022-05-13 23:47:19,"Bapak/Ibu, mohon maaf atas ketidaknyamanan yan...",2022-05-13 23:49:56,2.8.4
49998,0e1228cf-02d2-4ad6-bad0-2575442eb8c8,Siti Murniawati,https://play-lh.googleusercontent.com/a/ACg8oc...,Sangat menyesal saya telah mnggunakan aplikasi...,1,0,2.8.4,2022-05-13 23:42:15,"Bapak/Ibu, mohon maaf atas ketidaknyamanan yan...",2022-05-13 23:50:03,2.8.4


## 3. Data Cleaning dan Feature Engineering

Pada bagian ini data dibersihkan, dipilih kolom penting, dibuat fitur tambahan, dan diberi label sentimen sederhana berbasis skor.

In [ ]:
# Pilih kolom penting, bersihkan data, dan buat fitur sederhana
kolom_penting = [
    "reviewId", "userName", "score", "content", "at", "thumbsUpCount", "replyContent", "repliedAt"
    ]

df_clean = df_octo[kolom_penting].copy()
df_clean = df_clean.drop_duplicates(subset=["reviewId"])
df_clean = df_clean[df_clean["content"].notna()]
df_clean["content"] = df_clean["content"].astype(str).str.strip()
df_clean = df_clean[df_clean["content"] != ""]

# Konversi kolom tanggal ke datetime
df_clean["at"] = pd.to_datetime(df_clean["at"], errors="coerce")
df_clean["repliedAt"] = pd.to_datetime(df_clean["repliedAt"], errors="coerce")

# Feature engineering sederhana
df_clean["review_length"] = df_clean["content"].str.len()
df_clean["month"] = df_clean["at"].dt.to_period("M").astype(str)

# Label sentimen berbasis skor
def map_sentimen(score: int) -> str:
    if score >= 4:
        return "positif"
    if score == 3:
        return "netral"
    return "negatif"

df_clean["sentimen_label"] = df_clean["score"].apply(map_sentimen)

print("Ukuran data bersih:", df_clean.shape)
print("Distribusi sentimen:")
print(df_clean["sentimen_label"].value_counts())
df_clean.head()

## 4. Export Dataset

Simpan data mentah dan data bersih ke format CSV agar bisa dipakai untuk tahap preprocessing/modeling berikutnya.

In [ ]:
# Simpan data mentah dan data bersih
raw_path = "hasil_scraping_octo_raw.csv"
clean_path = "hasil_scraping_octo_clean.csv"

df_octo.to_csv(raw_path, index=False)
df_clean.to_csv(clean_path, index=False)
print(f"File tersimpan: {raw_path}, {clean_path}")

# Download otomatis jika dijalankan di Google Colab
try:
    from google.colab import files
    files.download(raw_path)
    files.download(clean_path)
except ImportError:
    print("Bukan environment Colab, lewati proses download otomatis.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Visualisasi EDA

Visualisasi ini membantu melihat distribusi rating, tren review dari waktu ke waktu, dan karakteristik panjang teks review.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1) Distribusi rating
plt.figure(figsize=(8, 4))
sns.countplot(data=df_clean, x="score", palette="Blues")
plt.title("Distribusi Rating Review OCTO Mobile")
plt.xlabel("Score")
plt.ylabel("Jumlah Review")
plt.show()

# 2) Tren jumlah review per bulan
review_per_bulan = df_clean.groupby("month").size().reset_index(name="jumlah_review")
review_per_bulan = review_per_bulan.sort_values("month")

plt.figure(figsize=(10, 4))
sns.lineplot(data=review_per_bulan, x="month", y="jumlah_review", marker="o", color="#2A6F97")
plt.title("Tren Jumlah Review per Bulan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah Review")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 3) Rata-rata panjang review per rating
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean, x="score", y="review_length", palette="crest")
plt.title("Panjang Review Berdasarkan Rating")
plt.xlabel("Score")
plt.ylabel("Jumlah Karakter")
plt.ylim(0, df_clean["review_length"].quantile(0.95))
plt.show()